### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import sys
sys.path.append('./utils')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator


This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


In [3]:
import mlflow
import os
os.environ['MLFLOW_TRACKING_URI'] = './mlruns'
import gc
import pandas as pd
from svg_processor import SVGSanitizer, SVGProcessor, svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator
from vllm import LLM, SamplingParams
import  re
from concurrent.futures import ThreadPoolExecutor

from tqdm import tqdm
tqdm.pandas()

class Model:
    def __init__(self):
        # MLflow experiment tracking setup
        self.experiment_name = "svg_score_test_45"
        mlflow.set_experiment(self.experiment_name)

        self.model_path = "./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
        self.model = LLM(
            model=self.model_path,
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.85
        )

        # Default generation parameters (can be overridden later in run_experiment)
        self.temperature = 0.5
        self.top_k = 40
        self.top_p = 0.95
        self.max_tokens = 1024

        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model
        gc.collect()

    def log_params_and_metrics(self, model_name, temperature, top_k, top_p, max_tokens, sl_score):
        """ Log parameters and metrics to MLflow """
        mlflow.log_param("model", model_name)
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("top_k", top_k)
        mlflow.log_param("top_p", top_p)
        mlflow.log_param("max_tokens", max_tokens)
        mlflow.log_metric("siglip_score", sl_score)
        
    def get_response(self, description, temperature, top_k, top_p, max_tokens):
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.

                ### Instruction:
                Please write a SVG code for the given input.

                ### Input:
                {}

                ### Response:
                """

        formatted_input = alpaca_prompt.format(description)
        sampling_params = SamplingParams(temperature=temperature, top_k=top_k, top_p=top_p, max_tokens=max_tokens)
        outputs = self.model.generate([formatted_input], sampling_params)

        # suitable for batch inputs as well        
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
        return generated_text

    def predict(self, description: str, temperature, top_k, top_p, max_tokens) -> str:
        output_decoded = self.get_response(description, temperature, top_k, top_p, max_tokens)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)

    def run_experiment(self, df, model_name, temperature=None, top_k=None, top_p=None, max_tokens=None):
        """ Log experiment and track prediction & evaluation """
        # Use passed parameters, or default to the instance's values
        temperature = temperature or self.temperature
        top_k = top_k or self.top_k
        top_p = top_p or self.top_p
        max_tokens = max_tokens or self.max_tokens
        
        with mlflow.start_run():
            # Generate SVG code
            df['svg'] = df['description'].apply(lambda x: self.predict(x, temperature, top_k, top_p, max_tokens))

            # Generate sl score
            df['sl_score'] = df.apply(lambda row: SVGMetricEvaluator().svg_metric(row['description'], row['svg']), axis=1)

            sl_score = df['sl_score'].mean()
            
            # Log parameters and metrics
            self.log_params_and_metrics(model_name, temperature, top_k, top_p, max_tokens, sl_score)
            
            return df, sl_score
    



INFO 04-08 22:31:02 [__init__.py:239] Automatically detected platform cuda.


In [4]:
#model instance 
model = Model()

2025/04/08 22:31:02 INFO mlflow.tracking.fluent: Experiment with name 'svg_score_test_45' does not exist. Creating a new experiment.


WARNING 04-08 22:31:02 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-08 22:31:07 [config.py:585] This model supports multiple tasks: {'score', 'generate', 'embed', 'classify', 'reward'}. Defaulting to 'generate'.
INFO 04-08 22:31:07 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-08 22:31:08 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', speculative_config=None, tokenizer='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_b

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-08 22:31:11 [loader.py:447] Loading weights took 2.08 seconds
INFO 04-08 22:31:11 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 2.228307 seconds
INFO 04-08 22:31:17 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/84ad9dae08/rank_0_0 for vLLM's torch.compile
INFO 04-08 22:31:17 [backends.py:425] Dynamo bytecode transform time: 5.90 s
INFO 04-08 22:31:17 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-08 22:31:22 [monitor.py:33] torch.compile takes 5.90 s in total
INFO 04-08 22:31:23 [kv_cache_utils.py:566] GPU KV cache size: 18,832 tokens
INFO 04-08 22:31:23 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 18.39x
INFO 04-08 22:31:40 [gpu_model_runner.py:1534] Graph capturing finished in 17 secs, took 0.43 GiB
INFO 04-08 22:31:40 [core.py:151] init engine (profile, create kv cache, warmup model) took 29.07 seconds


In [5]:
#load df & score
#df=pd.read_csv('./drawing-with-llms/train.csv',header=[0])
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df=df[['description','svg']]
df=df.iloc[:45]

In [6]:
temperature_list = [0.5,0.6,0.7,0.8,0.9]
top_k_list       = [60,70,80]
top_p_list       = [0.9,0.95,0.99]
max_tokens_list  = [1024]

print('No of experiments:',len(temperature_list)*len(top_k_list)*len(top_p_list)*len(max_tokens_list) )
print('Required GPU time (Hrs):',len(temperature_list)*len(top_k_list)*len(top_p_list)*len(max_tokens_list)*7/60 )

No of experiments: 45
Required GPU time (Hrs): 5.25


In [7]:
import time

for max_tokens in max_tokens_list:
    for top_p in top_p_list:
        for top_k in top_k_list:
            for temperature in temperature_list:

                model_name = re.sub(r'[^a-zA-Z0-9]', '_', model.model_path)
                
                current_time = time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())

                df_, sl_score = model.run_experiment(df, model_name, temperature=temperature, top_k=top_k, top_p=top_p,\
                                                     max_tokens=max_tokens)
                
                df_.to_csv(f'./io_files/{model_name}_{temperature}_{top_k}_{top_p}_{max_tokens}_\
                                                    {current_time}.csv')
                
                print('sl_score',sl_score)

Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.62s/it, est. speed input: 9.36 t
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.35s/it, est. speed input: 14.01 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.13s/it, est. speed input: 19.19 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.61s/it, est. speed input: 10.88 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.12s/it, est. speed input: 15.05 
Processed prompts: 100%|█| 1/1 [00:09<00:00,  9.39s/it, est. speed input: 6.71 t
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.04s/it, est. speed input: 20.39 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.16s/it, est. speed input: 28.77 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.80s/it, est. speed input: 12.50 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.68s/it, est. speed input: 23.56 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.92s/it, est. speed input: 20.53 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.14s/it, est. speed input: 14.98 
Processed prompts: 100%|█| 1

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
sl_score 0.36920154679631284


Processed prompts: 100%|█| 1/1 [00:10<00:00, 10.34s/it, est. speed input: 6.00 t


Failed to convert  'Vibrant autumn forest', due to invalid literal for int() with base 16: 'eg', Returning default SVG.


Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.80s/it, est. speed input: 21.76 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.63s/it, est. speed input: 22.78 
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.97s/it, est. speed input: 8.76 t
Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.47s/it, est. speed input: 7.32 t
Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.72s/it, est. speed input: 7.22 t
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.12s/it, est. speed input: 29.27 
Processed prompts: 100%|█| 1/1 [00:01<00:00,  1.54s/it, est. speed input: 40.27 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.99s/it, est. speed input: 12.04 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.28s/it, est. speed input: 19.19 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.18s/it, est. speed input: 14.36 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.31s/it, est. speed input: 14.39 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.12s/it, est. speed input: 28.76 
Processed prompts: 100%|█| 1

...

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
sl_score 0.5056823234106792


Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.19s/it, est. speed input: 7.57 t
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.44s/it, est. speed input: 25.00 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.06s/it, est. speed input: 19.59 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.72s/it, est. speed input: 10.67 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.18s/it, est. speed input: 11.98 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.90s/it, est. speed input: 12.85 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.15s/it, est. speed input: 28.86 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  3.00s/it, est. speed input: 20.70 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.62s/it, est. speed input: 12.99 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.96s/it, est. speed input: 21.29 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.05s/it, est. speed input: 19.65 
Processed prompts: 100%|█| 1/1 [00:11<00:00, 11.35s/it, est. speed input: 5.46 t
ERROR:root:SVG Parse Error: 

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
sl_score 0.44035203623239816


Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.82s/it, est. speed input: 7.03 t
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.43s/it, est. speed input: 17.78 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.62s/it, est. speed input: 10.67 
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.86s/it, est. speed input: 7.77 t
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.47s/it, est. speed input: 9.58 t
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.64s/it, est. speed input: 11.16 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.74s/it, est. speed input: 16.58 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.00s/it, est. speed input: 20.65 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.64s/it, est. speed input: 12.94 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.61s/it, est. speed input: 24.17 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.05s/it, est. speed input: 19.70 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.21s/it, est. speed input: 14.71 
Processed prompts: 100%|█| 1

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
sl_score 0.47272076742564323


In [8]:
#model.close_model()